In [1]:
# Importo las herramientas para subir archivos y leer CSV
from google.colab import files
from io import BytesIO
import re
import pandas as pd

# Subo juntos los dos CSV oficiales de AG News
archivos_subidos = files.upload()

# Identifico cada archivo incluso si su nombre tiene un sufijo como (2)
tablas = {}

for nombre, contenido in archivos_subidos.items():
    coincidencia = re.fullmatch(r"ag_news_(train|test)(?: \(\d+\))?\.csv", nombre)

    if coincidencia:
        conjunto = coincidencia.group(1)

        if conjunto in tablas:
            raise ValueError(f"Seleccioná un solo CSV para {conjunto}.")

        tablas[conjunto] = pd.read_csv(BytesIO(contenido))

if set(tablas) != {"train", "test"}:
    raise ValueError("Subí ag_news_train.csv y ag_news_test.csv juntos.")

# Guardo entrenamiento y prueba en tablas separadas
train_df = tablas["train"]
test_df = tablas["test"]

# Compruebo el tamaño y las columnas
print("Entrenamiento:", train_df.shape)
print("Prueba oficial:", test_df.shape)
print("Columnas:", train_df.columns.tolist())

# Muestro ejemplos del conjunto de entrenamiento
display(train_df.head())

Saving ag_news_test.csv to ag_news_test.csv
Saving ag_news_train.csv to ag_news_train.csv
Entrenamiento: (8000, 2)
Prueba oficial: (2000, 2)
Columnas: ['text', 'label']


,text,label
0,"Red Hat Appoints New CFO. September 2, 2004 --...",Sci_Tech
1,"Sprint, SBC In Hotspot Roaming Deal. Sprint an...",Sci_Tech
2,Radio Outage Grounds West Coast Flights. Airpo...,Business
3,Hollywood lawsuits to strike Net pirates. Holl...,Sci_Tech
4,Custis Takes Control. Family and Tony Robbins ...,Sports


In [2]:
%pip install -q spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 59.3 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import html
import re
import spacy

# Cargo el modelo de inglés para obtener la forma base de las palabras.
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])


def limpiar_texto(texto):
    # Reparo entidades incompletas como #39; y decodifico entidades HTML.
    texto = re.sub(r"(?<!&)#(\d+|[xX][0-9a-fA-F]+);", r"&#\1;", texto)
    texto = html.unescape(texto)

    # Quito etiquetas HTML y enlaces web.
    texto = re.sub(r"<[^>]+>", " ", texto)
    texto = re.sub(r"https?://\S+|www\.\S+", " ", texto)

    # Quito firmas duplicadas como (AP). AP -
    texto = re.sub(
        r"\((AP|AFP|Reuters)\)\s*\.\s*\1\s*[-–—]\s*",
        " ", texto, flags=re.IGNORECASE
    )

    # Quito bloques bursátiles como (GE.N: Quote, Profile, Research).
    texto = re.sub(
        r"\([^()]*:\s*Quote\s*,\s*Profile\s*,\s*Research\s*\)",
        " ", texto, flags=re.IGNORECASE
    )

    return re.sub(r"\s+", " ", texto).strip()


def preprocess_text(texto):
    documento = nlp(limpiar_texto(texto))
    lemas = [
        token.lemma_.lower()
        for token in documento
        if token.is_alpha
    ]
    return " ".join(lemas)


ejemplo = "<p>Technology news (AP). AP - The companies are growing.</p>"
print("Original:", ejemplo)
print("Procesado:", preprocess_text(ejemplo))

Original: <p>Technology news (AP). AP - The companies are growing.</p>
Procesado: technology news the company be grow


In [4]:
# Agrego una columna procesada y conservo el texto original.
train_df["text_clean"] = train_df["text"].apply(preprocess_text)

print("Noticias procesadas:", len(train_df))
print("Noticias vacías:", train_df["text_clean"].eq("").sum())
display(train_df[["text", "text_clean", "label"]].head())

Noticias procesadas: 8000
Noticias vacías: 0


,text,text_clean,label
0,"Red Hat Appoints New CFO. September 2, 2004 --...",red hat appoints new cfo september web host in...,Sci_Tech
1,"Sprint, SBC In Hotspot Roaming Deal. Sprint an...",sprint sbc in hotspot roaming deal sprint and ...,Sci_Tech
2,Radio Outage Grounds West Coast Flights. Airpo...,radio outage grounds west coast flights airpor...,Business
3,Hollywood lawsuits to strike Net pirates. Holl...,hollywood lawsuit to strike net pirate hollywo...,Sci_Tech
4,Custis Takes Control. Family and Tony Robbins ...,custis take control family and tony robbins ha...,Sports


In [5]:
from sklearn.model_selection import train_test_split

# Divido las 8.000 noticias oficiales de entrenamiento.
X_train, X_val, y_train, y_val = train_test_split(
    train_df["text_clean"],
    train_df["label"],
    test_size=0.20,
    random_state=42,
    stratify=train_df["label"]
)

print("Noticias para entrenar:", len(X_train))
print("Noticias para validar:", len(X_val))

Noticias para entrenar: 6400
Noticias para validar: 1600


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# Empiezo con palabras individuales, un máximo de 10.000 términos
# y las stop-words conservadas.
vectorizador_base = TfidfVectorizer(
    max_features=10_000,
    ngram_range=(1, 1),
    tokenizer=str.split,
    token_pattern=None,
    lowercase=False
)

# Aprendo el vocabulario y los pesos de TF-IDF solo en entrenamiento.
X_train_tfidf = vectorizador_base.fit_transform(X_train)

# Represento la validación con ese mismo vocabulario y esos mismos pesos.
X_val_tfidf = vectorizador_base.transform(X_val)

# Entreno el clasificador y predigo las categorías de validación.
modelo_base = LogisticRegression(max_iter=1000)
modelo_base.fit(X_train_tfidf, y_train)
predicciones_base = modelo_base.predict(X_val_tfidf)

# Mido el desempeño dando el mismo peso a cada categoría.
f1_base = f1_score(y_val, predicciones_base, average="macro")

print("Tamaño de la matriz de entrenamiento:", X_train_tfidf.shape)
print(f"F1 macro en validación: {f1_base:.4f}")

Tamaño de la matriz de entrenamiento: (6400, 10000)
F1 macro en validación: 0.8856


In [7]:
from itertools import product

# Uso la lista de stop-words de spaCy y conservo negaciones.
# También conservo "us" porque puede representar a Estados Unidos.
palabras_a_conservar = {"no", "not", "never", "none", "nor", "us"}
stop_words_filtradas = sorted(
    nlp.Defaults.stop_words - palabras_a_conservar
)

resultados = []

# Pruebo todas las combinaciones de los tres parámetros.
for limite, ngramas, filtrar in product(
    [10_000, 20_000],
    [(1, 1), (1, 2)],
    [False, True]
):
    vectorizador = TfidfVectorizer(
        max_features=limite,
        ngram_range=ngramas,
        stop_words=stop_words_filtradas if filtrar else None,
        tokenizer=str.split,
        token_pattern=None,
        lowercase=False
    )

    # Aprendo el vocabulario solo de las 6.400 noticias de entrenamiento.
    matriz_train = vectorizador.fit_transform(X_train)
    matriz_val = vectorizador.transform(X_val)

    modelo = LogisticRegression(max_iter=1000)
    modelo.fit(matriz_train, y_train)
    predicciones = modelo.predict(matriz_val)

    resultados.append({
        "max_features": limite,
        "ngram_range": ngramas,
        "filtrar_stop_words": filtrar,
        "columnas_tfidf": matriz_train.shape[1],
        "f1_macro_val": f1_score(y_val, predicciones, average="macro")
    })

# Ordeno de mayor a menor F1 macro en validación.
resultados_tfidf = (
    pd.DataFrame(resultados)
    .sort_values("f1_macro_val", ascending=False)
    .reset_index(drop=True)
)

display(resultados_tfidf.round({"f1_macro_val": 4}))

,max_features,ngram_range,filtrar_stop_words,columnas_tfidf,f1_macro_val
0,10000,"(1, 2)",True,10000,0.8941
1,20000,"(1, 2)",True,20000,0.8939
2,20000,"(1, 1)",True,15999,0.8933
3,10000,"(1, 1)",True,10000,0.8927
4,10000,"(1, 1)",False,10000,0.8856
5,20000,"(1, 1)",False,16248,0.8850
6,10000,"(1, 2)",False,10000,0.8787
7,20000,"(1, 2)",False,20000,0.8779
